# Testing with pytest and unittest

**Testing** verifies your code does what it's supposed to do.

### Why Test?
- Catch bugs before they reach production
- Safely refactor code (if tests pass, you didn't break anything)
- Document expected behavior
- Enable confident collaboration

### Tools
| Tool | Use Case |
|------|----------|
| `unittest` | Built-in, class-based, verbose |
| `pytest` | More powerful, less boilerplate, preferred |
| `unittest.mock` | Fake/mock external dependencies |

## 1. The Code We'll Test

First, let's define some functions to test throughout this notebook.

In [ ]:
# Functions and classes we'll write tests for

class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance
    
    @property
    def balance(self):
        return self._balance
    
    def deposit(self, amount):
        if amount <= 0:
            raise ValueError(f'Deposit must be positive, got {amount}')
        self._balance += amount
        return self._balance
    
    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError(f'Withdrawal must be positive, got {amount}')
        if amount > self._balance:
            raise ValueError(f'Insufficient funds')
        self._balance -= amount
        return self._balance
    
    def transfer(self, target, amount):
        self.withdraw(amount)
        target.deposit(amount)


def is_palindrome(s):
    cleaned = s.lower().replace(' ', '')
    return cleaned == cleaned[::-1]

def fizzbuzz(n):
    if n % 15 == 0: return 'FizzBuzz'
    if n % 3 == 0:  return 'Fizz'
    if n % 5 == 0:  return 'Buzz'
    return str(n)

print('Code ready for testing')

## 2. unittest — Built-in Testing Framework

In [ ]:
import unittest

class TestBankAccount(unittest.TestCase):
    
    def setUp(self):
        """Called before every test method — set up fresh state."""
        self.account = BankAccount('Alice', 1000)
        self.other = BankAccount('Bob', 500)
    
    def tearDown(self):
        """Called after every test method — clean up."""
        pass  # nothing to clean up here
    
    def test_initial_balance(self):
        self.assertEqual(self.account.balance, 1000)
    
    def test_deposit(self):
        self.account.deposit(200)
        self.assertEqual(self.account.balance, 1200)
    
    def test_deposit_negative_raises(self):
        with self.assertRaises(ValueError):
            self.account.deposit(-100)
    
    def test_withdraw(self):
        self.account.withdraw(300)
        self.assertEqual(self.account.balance, 700)
    
    def test_withdraw_insufficient_funds(self):
        with self.assertRaises(ValueError) as ctx:
            self.account.withdraw(5000)
        self.assertIn('Insufficient', str(ctx.exception))
    
    def test_transfer(self):
        self.account.transfer(self.other, 400)
        self.assertEqual(self.account.balance, 600)
        self.assertEqual(self.other.balance, 900)


class TestPalindrome(unittest.TestCase):
    
    def test_simple_palindrome(self):
        self.assertTrue(is_palindrome('racecar'))
    
    def test_not_palindrome(self):
        self.assertFalse(is_palindrome('hello'))
    
    def test_case_insensitive(self):
        self.assertTrue(is_palindrome('Racecar'))
    
    def test_with_spaces(self):
        self.assertTrue(is_palindrome('A man a plan a canal Panama'))


# Run tests in notebook
loader = unittest.TestLoader()
suite = unittest.TestSuite()
suite.addTests(loader.loadTestsFromTestCase(TestBankAccount))
suite.addTests(loader.loadTestsFromTestCase(TestPalindrome))

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

## 3. pytest — More Powerful, Less Boilerplate

pytest discovers and runs test functions automatically. No class required!

In [ ]:
# pytest style — plain functions, plain assert
# In a real project these would be in test_*.py files

# Simulating pytest behavior here
import traceback

def run_tests(*test_fns):
    """Simple test runner for demo purposes."""
    passed = failed = 0
    for fn in test_fns:
        try:
            fn()
            print(f'  PASS  {fn.__name__}')
            passed += 1
        except Exception as e:
            print(f'  FAIL  {fn.__name__}: {e}')
            failed += 1
    print(f'\n{passed} passed, {failed} failed')


# pytest-style tests
def test_fizzbuzz_divisible_by_3():
    assert fizzbuzz(9) == 'Fizz'

def test_fizzbuzz_divisible_by_5():
    assert fizzbuzz(10) == 'Buzz'

def test_fizzbuzz_divisible_by_15():
    assert fizzbuzz(15) == 'FizzBuzz'

def test_fizzbuzz_other():
    assert fizzbuzz(7) == '7'

def test_account_deposit_increases_balance():
    acc = BankAccount('test', 100)
    acc.deposit(50)
    assert acc.balance == 150

def test_account_deposit_zero_raises():
    acc = BankAccount('test')
    try:
        acc.deposit(0)
        assert False, 'Should have raised ValueError'
    except ValueError:
        pass  # expected

run_tests(
    test_fizzbuzz_divisible_by_3,
    test_fizzbuzz_divisible_by_5,
    test_fizzbuzz_divisible_by_15,
    test_fizzbuzz_other,
    test_account_deposit_increases_balance,
    test_account_deposit_zero_raises,
)

## 4. Mocking with `unittest.mock`

**Mocking** replaces real dependencies with fake ones for testing.

Use it when your code depends on:
- External APIs
- Databases
- File system
- Time/random values

In [ ]:
from unittest.mock import Mock, patch, MagicMock

# Mock object
mock = Mock()
mock.some_method(1, 2, 3)
print(mock.some_method.call_count)    # 1
print(mock.some_method.call_args)     # call(1, 2, 3)
mock.some_method.assert_called_with(1, 2, 3)  # no error = passed

# Mock with return value
mock_db = Mock()
mock_db.get_user.return_value = {'name': 'Alice', 'email': 'alice@example.com'}

user = mock_db.get_user(user_id=1)
print(f'Got user: {user}')

In [ ]:
from unittest.mock import patch
import datetime

# Real function that uses the current time
def get_greeting():
    hour = datetime.datetime.now().hour
    if hour < 12:
        return 'Good morning!'
    elif hour < 18:
        return 'Good afternoon!'
    else:
        return 'Good evening!'

# Test by mocking datetime.datetime.now()
class TestGreeting(unittest.TestCase):
    
    @patch('datetime.datetime')
    def test_morning(self, mock_dt):
        mock_dt.now.return_value.hour = 9  # force hour to 9
        self.assertEqual(get_greeting(), 'Good morning!')
    
    @patch('datetime.datetime')
    def test_afternoon(self, mock_dt):
        mock_dt.now.return_value.hour = 14
        self.assertEqual(get_greeting(), 'Good afternoon!')
    
    @patch('datetime.datetime')
    def test_evening(self, mock_dt):
        mock_dt.now.return_value.hour = 20
        self.assertEqual(get_greeting(), 'Good evening!')

# Note: @patch doesn't perfectly mock datetime in all cases, but shows the concept
print('Mocking example demonstrated')

## 5. Test-Driven Development (TDD)

TDD: **Red → Green → Refactor**
1. Write a failing test (Red)
2. Write minimum code to pass it (Green)
3. Refactor with confidence

Example: Build a password validator using TDD.

In [ ]:
# Step 1: Write tests FIRST
def test_password_too_short():
    assert not validate_password('abc') 

def test_password_needs_uppercase():
    assert not validate_password('password1!')

def test_password_needs_digit():
    assert not validate_password('Password!')

def test_password_needs_special():
    assert not validate_password('Password1')

def test_valid_password():
    assert validate_password('Password1!')


# Step 2: Implement the function
import re

def validate_password(password):
    """Password must be 8+ chars with uppercase, digit, and special character."""
    if len(password) < 8:
        return False
    if not re.search(r'[A-Z]', password):
        return False
    if not re.search(r'\d', password):
        return False
    if not re.search(r'[!@#$%^&*(),.?":{}|<>]', password):
        return False
    return True


# Step 3: Run tests
run_tests(
    test_password_too_short,
    test_password_needs_uppercase,
    test_password_needs_digit,
    test_password_needs_special,
    test_valid_password,
)

## Mini-Project: Full Test Suite for BankAccount

A comprehensive test suite covering all edge cases.

In [ ]:
import unittest

class TestBankAccountFull(unittest.TestCase):
    
    def setUp(self):
        self.alice = BankAccount('Alice', 1000)
        self.bob = BankAccount('Bob', 500)
    
    # --- Initialization ---
    def test_initial_balance(self):
        self.assertEqual(self.alice.balance, 1000)
    
    def test_default_balance_is_zero(self):
        acc = BankAccount('Charlie')
        self.assertEqual(acc.balance, 0)
    
    # --- Deposit ---
    def test_deposit_increases_balance(self):
        self.alice.deposit(200)
        self.assertEqual(self.alice.balance, 1200)
    
    def test_deposit_returns_new_balance(self):
        new_balance = self.alice.deposit(200)
        self.assertEqual(new_balance, 1200)
    
    def test_deposit_zero_raises_value_error(self):
        with self.assertRaises(ValueError):
            self.alice.deposit(0)
    
    def test_deposit_negative_raises_value_error(self):
        with self.assertRaises(ValueError):
            self.alice.deposit(-50)
    
    # --- Withdraw ---
    def test_withdraw_decreases_balance(self):
        self.alice.withdraw(300)
        self.assertEqual(self.alice.balance, 700)
    
    def test_withdraw_entire_balance(self):
        self.alice.withdraw(1000)
        self.assertEqual(self.alice.balance, 0)
    
    def test_withdraw_overdraft_raises(self):
        with self.assertRaises(ValueError) as ctx:
            self.alice.withdraw(1001)
        self.assertIn('Insufficient', str(ctx.exception))
    
    def test_withdraw_zero_raises(self):
        with self.assertRaises(ValueError):
            self.alice.withdraw(0)
    
    # --- Transfer ---
    def test_transfer_moves_money(self):
        self.alice.transfer(self.bob, 200)
        self.assertEqual(self.alice.balance, 800)
        self.assertEqual(self.bob.balance, 700)
    
    def test_transfer_overdraft_raises(self):
        with self.assertRaises(ValueError):
            self.alice.transfer(self.bob, 5000)
    
    def test_transfer_leaves_source_unchanged_on_failure(self):
        try:
            self.alice.transfer(self.bob, 5000)
        except ValueError:
            pass
        self.assertEqual(self.alice.balance, 1000)  # unchanged


# Run all tests
loader = unittest.TestLoader()
suite = loader.loadTestsFromTestCase(TestBankAccountFull)
runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)
print(f'\n{result.testsRun} tests, {len(result.failures)} failures, {len(result.errors)} errors')